# 5-1. PEFT — 파라미터 효율적 튜닝 (LoRA & QLoRA)

---

## 목차

| # | 내용 |
|:---:|------|
| 0 | **환경 설정** — Unsloth 설치, 모델 로딩 |
| 1 | **왜 Fine-tuning이 필요한가** — 사전학습 모델의 한계 |
| 2 | **Full Fine-tuning의 문제** — 메모리 폭발을 직접 계산 |
| 3 | **LoRA의 원리** — 1%만 학습하는 마법 |
| 4 | **QLoRA** — 5-2에서 배운 양자화 + LoRA의 결합 |
| 5 | **실전: QLoRA 학습** — 데이터 준비부터 학습까지 |
| 6 | **정리** |

<br>

> **📌 이 실습은**
>
> 개념을 이해한 뒤, 자기주도 실습(`실습_5-1`)에서 직접 코드를 작성하게 된다.
>
> **🔗 5-2 연결**: 지난 시간에 배운 **INT4/NF4 양자화**가 이번 QLoRA에서 그대로 사용된다.
> `"가볍게 만든 모델 위에 소수의 파라미터만 학습시킨다"`는 것이 핵심이다.


---

## 0. 환경 설정

### Unsloth란?

**Unsloth**는 LLM 파인튜닝에 최적화된 오픈소스 프레임워크이다.
일반적인 HuggingFace PEFT 대비 **학습 속도 최대 30배 향상, 메모리 30% 절감**을 달성한다.

| 항목 | 표준 HuggingFace PEFT | Unsloth |
|------|:---:|:---:|
| 학습 속도 | 기준 | **최대 30배 빠름** |
| 메모리 | 기준 | **30% 절감** |
| 코드 복잡도 | 복잡 (BitsAndBytesConfig + PeftModel 등) | **간편 (FastModel 한 줄)** |

내부적으로 수작업 최적화된 GPU 커널을 사용하며, QLoRA/LoRA를 간편하게 적용할 수 있다.
이 실습에서 Unsloth를 사용하는 이유는 **16GB VRAM 환경에서도 빠르고 안정적으로 학습**할 수 있기 때문이다.

### Gemma란? 왜 gemma-3-1b를 사용하는가?

**Gemma**는 Google DeepMind가 공개한 오픈소스 LLM 시리즈이다.
"Gemini의 경량 오픈소스 버전"으로, 연구 및 교육 목적으로 자유롭게 사용할 수 있다.

| 모델 | 파라미터 | 특징 |
|------|:---:|------|
| gemma-3-1b | **10억 개** | 가벼워서 16GB GPU에서 학습 가능 |
| gemma-3-4b | 40억 개 | 중간 크기 |
| gemma-3-12b | 120억 개 | 고성능, 고사양 GPU 필요 |
| gemma-3-27b | 270억 개 | 최고 성능, A100급 필요 |

**gemma-3-1b를 선택한 이유**:
1. 16GB VRAM에서 4-bit QLoRA 학습이 가능한 크기
2. 학습 시간이 짧아 수업 시간 내 결과 확인 가능
3. HuggingFace 접근 승인(Gated) 없이 바로 사용 가능
4. Unsloth에서 최적화된 4-bit 버전을 제공


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 역할: Unsloth 및 학습에 필요한 패키지 설치
# - unsloth: 최적화된 LLM 파인튜닝 프레임워크
# - peft: LoRA 등 PEFT 기법 제공
# - trl: SFTTrainer(지도 학습 미세조정 트레이너)
# - datasets: HuggingFace 데이터셋 로딩
# ═══════════════════════════════════════════════════════════

!pip install -q unsloth
!pip install -q --upgrade typing_extensions
!pip install -q \
    transformers>=4.55.0 peft>=0.10.0 trl>=0.8.0 \
    datasets>=3.0.0 accelerate>=0.30.0 bitsandbytes>=0.43.0 tokenizers>=0.20.0


In [ ]:
from unsloth import FastModel
import torch
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: Unsloth로 4-bit 양자화 모델을 로딩
#
# [이 셀에서 하는 것]
# Gemma-3-1B 모델을 NF4(4-bit) 양자화 상태로 GPU에 로딩한다.
# 5-2에서 BitsAndBytesConfig를 직접 설정했던 것을
# Unsloth는 FastModel.from_pretrained() 한 줄로 처리한다.
#
# [모델 이름 해석]
# 'unsloth/gemma-3-1b-it-unsloth-bnb-4bit'
#  ├─ unsloth/     → Unsloth가 최적화한 버전
#  ├─ gemma-3      → Google의 Gemma 3세대
#  ├─ 1b           → 10억(1 Billion) 파라미터
#  ├─ it           → Instruction-Tuned (대화형으로 사전학습됨)
#  └─ bnb-4bit     → bitsandbytes 4-bit 양자화 적용
#
# [5-2 연결]
# load_in_4bit=True → 5-2에서 배운 NF4 양자화가 적용된다.
# 5-2에서는 "추론용"으로 양자화했지만,
# 여기서는 "학습의 베이스 모델"로 양자화를 사용한다.
#
# [확인 포인트]
# GPU 메모리가 ~1GB 수준으로 나오면 정상이다.
# 4-bit 양자화 덕분에 10억 파라미터 모델이 ~1GB에 로딩된다.
# (FP16이라면 ~2GB, FP32라면 ~4GB가 필요했을 것이다.)
# ═══════════════════════════════════════════════════════════

max_seq_length = 1024  # 시퀀스 최대 길이 (메모리 효율을 위해 제한)

# ★ 핵심: 4-bit 양자화 모델 로딩 (Unsloth가 내부적으로 NF4 설정)
model, tokenizer = FastModel.from_pretrained(
    model_name='unsloth/gemma-3-1b-it-unsloth-bnb-4bit',
    max_seq_length=max_seq_length,
    load_in_4bit=True,   # ← 5-2에서 배운 INT4/NF4 양자화!
)

print(f'모델 로딩 완료: Gemma-3-1B (4-bit 양자화)')
if torch.cuda.is_available():
    mem = torch.cuda.memory_allocated() / 1024**3
    print(f'GPU 메모리: {mem:.2f} GB')
    print(f'→ 10억 파라미터 모델이 4-bit 양자화 덕분에 ~1GB에 로딩되었다.')


---

## 1. 왜 Fine-tuning이 필요한가?

### 1-1. 사전학습 모델의 한계

GPT, Gemma, Llama 같은 LLM은 인터넷의 방대한 텍스트로 **사전학습(Pre-training)**되어 있다.
일반적인 질문에는 잘 답하지만, **특정 도메인이나 형식**에는 약하다.

| 요청 | 사전학습 모델 | Fine-tuning된 모델 |
|------|:---:|:---:|
| "서울 날씨 알려줘" | ✅ 잘 답함 | ✅ 잘 답함 |
| 체스 기보 → 다음 수 예측 | ❌ 체스 규칙 모름 | ✅ 학습 후 정확히 예측 |
| 사내 규정 기반 답변 | ❌ 사내 정보 없음 | ✅ 사내 데이터로 학습 |
| 특정 말투/형식 | ❌ 범용 말투 | ✅ 원하는 형식으로 출력 |

**Fine-tuning** = 사전학습된 모델을 **특정 작업/도메인에 맞게 추가 학습**시키는 것

> **💡 비유: 의대 졸업생과 전문의**
>
> - **사전학습 모델** = 의대를 졸업한 일반의. 기본 의학 지식은 있지만 전문 수술은 못 함
> - **Fine-tuning된 모델** = 전문의 수련을 마친 외과 전문의. 특정 분야에서 전문적 역량 발휘

### 1-2. 이 실습의 목표

Gemma-3-1B 모델을 **체스 다음 수 예측** 전문가로 만든다.

### 1-3. 체스 기보(棋譜) 읽는 법

체스 기보는 체스 경기의 수순을 기록한 것이다. 각 기호의 의미:

| 기호 | 의미 | 설명 |
|:---:|------|------|
| e4 | 폰(Pawn)을 e4 칸으로 이동 | 소문자 = 폰 |
| Nf3 | 나이트(Knight)를 f3 칸으로 이동 | N = Knight |
| Bb5 | 비숍(Bishop)을 b5 칸으로 이동 | B = Bishop |

이 실습에서 사용하는 기보 `1. e4 e5 2. Nf3 Nc6 3. Bb5`는
`Ruy Lopez Opening(루이 로페스 오프닝)`이라 불리는 유명한 체스 시작 수순이다.
16세기 스페인 사제 Ruy López de Segura가 체계화한 것으로,
현재까지도 프로 체스에서 가장 많이 사용되는 오프닝 중 하나이다.

```
1. e4 e5     →  백: 폰을 e4로 / 흑: 폰을 e5로  (중앙 제어)
2. Nf3 Nc6   →  백: 나이트를 f3으로 / 흑: 나이트를 c6으로  (전개)
3. Bb5       →  백: 비숍을 b5로  (흑 나이트를 위협)
   → 다음 흑의 최선 수: a6 ("Morphy Defense" — 비숍을 밀어내는 수)
```

학습 후 모델이 이 기보를 보고 `"a6"`을 정확히 예측하면 성공이다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: Fine-tuning "전" 베이스 모델의 답변을 기록 (Before)
#
# [이 셀에서 하는 것]
# 체스 기보를 입력했을 때 학습 전 모델이 어떤 답변을 하는지 확인한다.
# 이 결과를 response_before에 저장해 두고,
# 마지막 셀에서 학습 후 결과(After)와 비교한다.
#
# [확인 포인트]
# 학습 전 모델은 체스 전문 지식이 없으므로:
# - "a6" 같은 정확한 수를 예측하지 못하고
# - 체스에 대한 일반적인 설명이나 엉뚱한 답변을 할 가능성이 높다.
# 이것이 "Fine-tuning이 왜 필요한가"의 직접적 증거이다.
#
# [체스 기보 해설]
# "1. e4 e5 2. Nf3 Nc6 3. Bb5" = Ruy Lopez Opening(루이 로페스)
# 이 기보의 정답(흑의 최선 수): a6 (Morphy Defense)
# ═══════════════════════════════════════════════════════════

# 테스트 입력: Ruy Lopez Opening
test_input = '1. e4 e5 2. Nf3 Nc6 3. Bb5'

messages = [
    {'role': 'user', 'content': f'Predict the next best move: {test_input}'}
]

# apply_chat_template: Gemma-3의 대화 형식으로 변환
# add_generation_prompt=True → 모델이 답변을 생성하도록 유도
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

# 추론 실행
inputs = tokenizer(text, return_tensors='pt').to('cuda')
with torch.no_grad():  # 추론 시 그래디언트 계산 불필요
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)

# 응답 추출 (입력 부분 제외, 새로 생성된 토큰만)
response_before = tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
)

print(f'입력 기보: {test_input}')
print(f'(해설: Ruy Lopez Opening — 흑의 최선 수는 a6)')
print(f'\n학습 전 응답: {response_before[:200]}')
print(f'\n→ 체스 전문 지식 없이 범용적인 답변을 하는 것을 확인하라.')
print(f'  Fine-tuning 후 이 답변이 "a6"으로 바뀌는지 마지막에 비교한다.')


---

## 2. Full Fine-tuning의 문제 — 메모리 폭발

### 2-1. Full Fine-tuning이란?

가장 직관적인 학습 방법: 모델의 **모든 파라미터를 업데이트**한다.

하지만 심각한 문제가 있다. 학습에는 **모델 가중치 + Gradient + Optimizer 상태**를 모두 GPU에 올려야 한다.

### 2-2. 학습 시 메모리 구성

```
[Full Fine-tuning 메모리 구성 — 1B 모델, FP32 기준]

모델 파라미터:     1B × 4바이트 =  4GB  ← 가중치 자체
Gradient:         1B × 4바이트 =  4GB  ← 각 파라미터의 기울기
Optimizer (AdamW): 1B × 8바이트 =  8GB  ← momentum + variance (2개)
Activations:       배치 크기에 따라   4~16GB  ← 중간 계산 결과
─────────────────────────────────
합계:                           약 20~32GB
```

> **💡 Optimizer가 왜 8바이트나 쓰는가?**
>
> AdamW는 각 파라미터에 대해 `momentum(1차 모멘트)`과 `variance(2차 모멘트)`를
> FP32로 저장한다. 파라미터 1개당 4바이트 × 2 = 8바이트.
> 즉 Optimizer 상태만으로 모델 가중치의 **2배** 메모리를 차지한다.

우리 GPU는 **16GB VRAM**이다. 1B 모델의 Full Fine-tuning에만 20~32GB가 필요하니
**사실상 불가능**하다. 7B 모델이면? 최소 100GB 이상이다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: Full Fine-tuning에 필요한 메모리를 직접 계산
#
# [계산 공식]
# 모델: 파라미터 수 × 4바이트(FP32)
# Gradient: 파라미터 수 × 4바이트
# Optimizer(AdamW): 파라미터 수 × 8바이트 (momentum + variance)
# Activation: 모델 크기의 1~4배 (배치 크기에 비례)
#
# [예상 결과]
# 1B 모델: ~20GB → 16GB GPU에서 불가능
# 7B 모델: ~100GB+ → A100 2장 이상 필요
# ═══════════════════════════════════════════════════════════

total_params = sum(p.numel() for p in model.parameters())

def calc_full_ft_memory(params, label):
    """Full Fine-tuning에 필요한 총 메모리를 계산한다."""
    model_mem = params * 4 / 1024**3     # FP32 가중치
    grad_mem = params * 4 / 1024**3      # Gradient (FP32)
    optim_mem = params * 8 / 1024**3     # AdamW (momentum + variance, FP32)
    act_mem = model_mem * 2              # Activation (추정: 모델의 ~2배)
    total = model_mem + grad_mem + optim_mem + act_mem
    print(f'\n[{label}] Full Fine-tuning 메모리 요구량:')
    print(f'  모델 파라미터:  {model_mem:6.1f} GB')
    print(f'  Gradient:     {grad_mem:6.1f} GB')
    print(f'  Optimizer:    {optim_mem:6.1f} GB (AdamW = momentum + variance)')
    print(f'  Activation:   {act_mem:6.1f} GB (추정)')
    print(f'  ──────────────────────')
    print(f'  합계:         {total:6.1f} GB')
    return total

print(f'현재 모델 파라미터: {total_params:,}개 ({total_params/1e9:.2f}B)')

mem_1b = calc_full_ft_memory(total_params, 'Gemma-3-1B')
mem_7b = calc_full_ft_memory(7_000_000_000, '7B 모델 (참고)')

print(f'\n─────────────────────────────')
print(f'우리 GPU VRAM: 16 GB')
print(f'Gemma-3-1B Full FT: {mem_1b:.1f} GB → {"❌ 불가능" if mem_1b > 16 else "✅ 가능"}')
print(f'7B 모델 Full FT: {mem_7b:.1f} GB → ❌ A100 2장 이상 필요')
print(f'\n→ Full Fine-tuning은 16GB GPU에서 사실상 불가능하다.')
print(f'  해결책: 전체가 아닌 "일부 파라미터만" 학습하는 PEFT → 다음 챕터')


---

## 3. LoRA의 원리 — 1%만 학습하는 마법

![image_A](https://i.ibb.co/kV5rS1KW/image-A.png)

### 3-1. PEFT란?

**PEFT (Parameter-Efficient Fine-Tuning)** = 전체 파라미터 중 **극소수만 학습**하는 기법

- 사전학습된 가중치는 **고정(freeze)** ❄️
- 적은 수의 **추가 파라미터(adapter)**만 학습 🔥
- Full FT와 비슷한 성능을 달성하면서 메모리를 대폭 절감

PEFT의 대표 기법이 바로 `LoRA (Low-Rank Adaptation)`이다.

### 3-2. LoRA의 핵심 아이디어

LoRA의 핵심은 `"큰 행렬을 직접 수정하지 말고, 작은 행렬 두 개로 변화량을 표현하자"`이다.

> **💡 비유: 건물 리모델링**
>
> - **Full FT** = 건물 전체를 허물고 새로 짓기 (비용 막대)
> - **LoRA** = 건물은 그대로 두고, 필요한 방만 리모델링 (비용 최소)

수식으로 표현하면:

```
기존: h = W × x              (W: d×d 행렬, 전체 학습)
LoRA: h = W × x + B × A × x  (W: 고정, B×A만 학습)
```

### 3-3. d × r, r × d 가 의미하는 것

```
┌─────────────────────────────────────────┐
│         W (사전학습 가중치)               │ ← 고정 ❄️ (수십억 개)
│              d × d                      │
└─────────────────────────────────────────┘
                    +
┌──────────┐   ┌──────────┐
│    B     │ × │    A     │  ← 학습 🔥 (수십만 개)
│  d × r   │   │  r × d   │
└──────────┘   └──────────┘
```

여기서 **d**와 **r**은:

| 기호 | 의미 | 예시 |
|:---:|------|------|
| **d** | 원본 가중치 행렬의 **차원** (행/열 크기) | Gemma-1B: d = 2048 |
| **r** | LoRA의 **rank** (축소할 차원) | 보통 r = 8 또는 16 |

**d × r** = "d개의 행, r개의 열"을 가진 행렬이라는 뜻이다.

- **B 행렬 (d × r)**: 2048행 × 8열 = 16,384개 파라미터
- **A 행렬 (r × d)**: 8행 × 2048열 = 16,384개 파라미터
- **B × A 결과**: (d × r) × (r × d) = **d × d** → W와 같은 크기!

즉, 작은 행렬 두 개(B, A)를 곱하면 원본 W와 같은 크기의 "변화량"이 만들어진다.
하지만 파라미터 수는 **d × d**가 아니라 **2 × d × r**만 필요하므로 압도적으로 적다.

### 3-4. 숫자로 비교

d = 4096 (일반적인 LLM 차원), r = 8인 경우:

| 구분 | 계산 | 파라미터 수 | 비율 |
|------|------|:---:|:---:|
| **W (전체)** | 4096 × 4096 | **16,777,216** | 100% |
| **B + A (LoRA)** | (4096 × 8) + (8 × 4096) | **65,536** | **0.39%** |

전체의 **0.4%만 학습**해도 충분한 성능을 달성할 수 있다.
r을 키우면(16, 32) 표현력이 올라가지만 메모리도 비례하여 증가한다.

### 3-5. LoRA 핵심 하이퍼파라미터

| 파라미터 | 의미 | 권장값 |
|---------|------|:---:|
| **r (rank)** | 저차원 행렬의 차원. 클수록 표현력↑, 메모리↑ | 8, 16, 32 |
| **lora_alpha** | adapter 출력의 스케일링 값. α/r을 곱하여 학습률 조절 | r × 2 |
| **target_modules** | LoRA를 적용할 레이어 | q_proj, k_proj, v_proj 등 |
| **lora_dropout** | 과적합 방지 드롭아웃 | 0.0 |

<br>

> **💡 lora_alpha와 r의 관계**
>
> adapter 출력에 **α/r**을 곱하여 영향력을 조절한다.
> r=8, alpha=16이면 **16/8 = 2배**로 adapter 영향력이 증폭된다.
> alpha를 r의 2배로 설정하는 것이 일반적인 관행이다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: LoRA를 모델에 적용하고 학습 파라미터 비율 확인
#
# [이 셀에서 하는 것]
# 1. LoRA 하이퍼파라미터(r, alpha, target_modules)를 설정
# 2. FastModel.get_peft_model()로 LoRA를 모델에 적용
# 3. 전체 파라미터 중 학습되는 비율을 확인
#
# [확인 포인트]
# - 학습 비율이 약 1~2%로 나오면 정상이다.
# - 나머지 98~99%의 사전학습 가중치는 고정(freeze)된 상태이다.
#
# [target_modules 설명]
# LLM의 Transformer 블록은 크게 Attention과 MLP로 구성된다.
# - Attention: q_proj(Query), k_proj(Key), v_proj(Value), o_proj(Output)
# - MLP: gate_proj, up_proj, down_proj
# 이 레이어들에 LoRA adapter를 적용하면 모델의 핵심 연산을 커버한다.
# ═══════════════════════════════════════════════════════════

# ★ 핵심: LoRA 하이퍼파라미터 설정
r = 8             # rank: 저차원 행렬의 차원 (작을수록 경량)
lora_alpha = 16   # 스케일링: r × 2 권장 (α/r = 16/8 = 2배 증폭)
lora_dropout = 0.0

# LoRA를 적용할 레이어들
target_modules = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',  # Attention 레이어
    'gate_proj', 'up_proj', 'down_proj',       # MLP 레이어
]

# LoRA 적용 (Unsloth 최적화)
model = FastModel.get_peft_model(
    model,
    r=r,
    target_modules=target_modules,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
)

# ========== 학습 파라미터 비율 확인 ==========
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
ratio = trainable / total * 100

print(f'총 파라미터:  {total:>15,}개')
print(f'학습 파라미터: {trainable:>15,}개')
print(f'학습 비율:    {ratio:.2f}%')
print(f'\n→ 전체의 약 {ratio:.1f}%만 학습한다. (나머지 {100-ratio:.1f}%는 고정)')
print(f'  챕터 2에서 계산한 Full FT 메모리 문제가 이것으로 해결된다.')


---

## 4. QLoRA — 양자화 + LoRA의 결합

![image_B](https://i.ibb.co/Gf1yTfV3/image-B.png)

### 4-1. 5-2 복습: 양자화의 역할

5-2에서 배운 핵심을 떠올려 보자:
- **양자화** = 가중치의 비트 수를 줄여 메모리 절감 (FP16 → INT4)
- **NF4** = 정규분포에 최적화된 4-bit 양자화
- **PTQ** = 학습 완료 후 양자화 (5-2에서 사용)

### 4-2. QLoRA = Q(양자화된 모델) + LoRA(저차원 Adapter)

QLoRA는 이 두 가지를 **결합**한 것이다:

```
┌─────────────────────────────────────────┐
│   Base Model (사전학습 가중치)            │ ← 4-bit NF4로 양자화 ❄️
│   INT4 (0.5GB per 1B params)            │    5-2에서 배운 것!
└─────────────────────────────────────────┘
                    +
┌──────────┐   ┌──────────┐
│    B     │ × │    A     │  ← FP16으로 학습 🔥
│  d × r   │   │  r × d   │    전체의 ~1%
└──────────┘   └──────────┘
```

| 구분 | LoRA | QLoRA |
|------|:---:|:---:|
| Base Model | FP16 (~2GB/1B) | **INT4 (~0.5GB/1B)** |
| Adapter | FP16 | FP16 (동일) |
| 7B 모델 메모리 | ~14GB | **~4GB** |
| 16GB GPU에서 7B | ❌ 빠듯 | ✅ 여유 |

<br>

> **💡 QLoRA의 핵심 기술 3가지 (5-2 연결)**
>
> 1. **NF4 양자화**: 5-2에서 배운 그것. Base Model을 4-bit로 압축
> 2. **Double Quantization**: 양자화 상수까지 다시 양자화 (5-2의 `bnb_4bit_use_double_quant=True`)
> 3. **Paged Optimizers**: GPU 메모리 부족 시 CPU로 자동 스왑

### 4-3. Unsloth의 역할

| 구분 | 표준 QLoRA | Unsloth QLoRA |
|------|:---:|:---:|
| 학습 속도 | 기준 | **최대 30배 빠름** |
| 메모리 | 기준 | **30% 절감** |
| 코드 복잡도 | 복잡 | **간편 (FastModel)** |


---

## 5. 실전: QLoRA 학습

![image_C](https://i.ibb.co/C33xtCxD/image-C.png)

### 5-1. 학습 파이프라인 전체 흐름

```
① 데이터 로드 → ② Chat Template 변환 → ③ Trainer 설정 → ④ 학습 → ⑤ 저장/추론
```

### 5-2. 데이터셋: ChessInstruct

| 항목 | 내용 |
|------|------|
| 이름 | Thytu/ChessInstruct |
| 구조 | task(지시), input(체스 기보), expected_output(다음 수) |
| 크기 | 5,000건 (학습용) |

### 5-3. Chat Template이란?

LLM은 특정 **대화 형식**을 기대한다. 원본 데이터를 이 형식에 맞게 변환해야 한다.

```
[원본 데이터]                          [Chat Template 변환 후]
task: "Predict the next move"    →    system: "Predict the next move"
input: "1. e4 e5 2. Nf3"        →    user: "1. e4 e5 2. Nf3"
expected_output: "Nc6"           →    assistant: "Nc6"
```


In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: 데이터 로드 + Chat Template 변환
#
# [이 셀에서 하는 것]
# 1. ChessInstruct 데이터셋 5,000건을 HuggingFace에서 로드
# 2. 원본 3개 필드(task, input, expected_output)를
#    LLM이 이해하는 conversations 형식(system/user/assistant)으로 변환
# 3. Gemma-3의 chat template을 적용하여 실제 학습 텍스트로 변환
#
# [변환 흐름]
# 원본: {task: "Predict...", input: "1. e4 e5", expected_output: "Nc6"}
#   ↓ convert_to_chatml()
# Chat: {conversations: [{role:system,...}, {role:user,...}, {role:assistant,...}]}
#   ↓ formatting_prompts_func()
# 텍스트: "<start_of_turn>user\n1. e4 e5<end_of_turn>\n<start_of_turn>model\nNc6"
#
# [add_generation_prompt=False인 이유]
# 학습 시에는 assistant 응답이 이미 포함되어 있으므로
# 모델이 새로 생성할 필요가 없다. (추론 시에는 True로 설정)
# ═══════════════════════════════════════════════════════════

# 학습 데이터 로드 (5,000건)
dataset = load_dataset('Thytu/ChessInstruct', split='train[:5000]')
print(f'데이터 로드: {len(dataset)}건')
print(f'샘플: {dataset[0]}')

# ========== 1단계: Chat Template 변환 ==========
def convert_to_chatml(example):
    """원본 데이터를 conversations 형식(system/user/assistant)으로 변환한다."""
    return {
        'conversations': [
            {'role': 'system', 'content': example['task']},      # 작업 지시
            {'role': 'user', 'content': example['input']},       # 체스 기보
            {'role': 'assistant', 'content': example['expected_output']},  # 정답 수
        ]
    }

dataset = dataset.map(convert_to_chatml)

# ========== 2단계: 텍스트 포맷팅 ==========
def formatting_prompts_func(examples):
    """conversations를 Gemma-3 chat template 텍스트로 변환한다."""
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False,
            add_generation_prompt=False  # 학습 시: 응답이 이미 있으므로 False
        ).removeprefix('<bos>')  # 중복 BOS 토큰 제거
        for convo in examples['conversations']
    ]
    return {'text': texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f'\n변환 완료. 샘플 텍스트:')
print(dataset[0]['text'][:300])


In [ ]:
# ⏱️ [학습 시간 안내 — 이 셀은 오래 걸린다!]
# - 강의장 환경: 약 15~20분 소요 예상
# → 학습이 진행되는 동안 아래 이론 설명을 읽으며 기다리면 된다.

from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: SFTTrainer 설정 + 학습 실행
#
# [이 셀에서 하는 것]
# 1. SFTTrainer(지도 학습 미세조정 트레이너)를 설정한다.
# 2. train_on_responses_only()로 "답변 부분만" 학습하도록 설정한다.
# 3. trainer.train()으로 실제 학습을 실행한다.
#
# [SFTTrainer란?]
# Supervised Fine-Tuning(지도 학습 미세조정) 전용 트레이너이다.
# trl(Transformer Reinforcement Learning) 라이브러리에서 제공하며,
# LoRA/QLoRA와 자연스럽게 연동된다.
#
# [train_on_responses_only — 왜 필요한가?]
# 데이터에는 "질문(instruction)"과 "답변(response)"이 모두 포함되어 있다.
# 우리가 학습시키고 싶은 것은 "답변을 잘 하는 법"이지,
# "질문을 잘 읽는 법"이 아니다.
# train_on_responses_only()는 답변 부분의 Loss만 계산하여 학습 효율을 높인다.
#
# [주요 학습 설정 해설]
# - per_device_train_batch_size=2: GPU당 한 번에 처리하는 데이터 수
# - gradient_accumulation_steps=4: 4번 누적 후 업데이트 → 실효 배치 크기 = 2×4 = 8
#   (메모리가 부족할 때 배치 크기를 키우는 트릭)
# - num_train_epochs=1: 전체 데이터를 1번만 학습 (이론 실습이므로 빠르게)
# - fp16/bf16: 학습 중 16비트 정밀도 사용 (메모리 절감)
# - logging_steps=10: 10 step마다 Loss를 출력 (학습 진행 확인)
#
# [학습 결과 해석 가이드]
# 학습이 완료되면 다음 값이 출력된다:
# - 학습 후 메모리: ~12GB → 16GB VRAM의 약 75% 사용 (정상)
# - Loss: ~0.6 → 체스 다음 수 예측의 불확실성 수준
#   (Loss가 0에 가까울수록 좋지만, 체스는 여러 좋은 수가 있으므로
#    0.5~0.8 수준이면 충분히 학습된 것이다)
#
# ═══════════════════════════════════════════════════════════

# Trainer 설정
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field='text',
        output_dir='outputs',
        per_device_train_batch_size=2,      # GPU당 배치 크기
        gradient_accumulation_steps=4,       # 4번 누적 → 실효 배치 8
        learning_rate=5e-5,                  # 학습률
        num_train_epochs=1,                  # 1에폭 (이론 실습용)
        max_seq_length=max_seq_length,       # 시퀀스 길이 제한
        fp16=not torch.cuda.is_bf16_supported(),  # GPU가 bf16 미지원이면 fp16
        bf16=torch.cuda.is_bf16_supported(),      # GPU가 bf16 지원하면 bf16
        logging_steps=10,                    # 10 step마다 Loss 출력
        seed=42,
    ),
)

# ★ 핵심: Response(답변) 부분만 학습하도록 설정
# instruction_part: 질문이 시작되는 토큰 (학습 제외)
# response_part: 답변이 시작되는 토큰 (학습 대상)
trainer = train_on_responses_only(
    trainer,
    instruction_part='<start_of_turn>user\n',
    response_part='<start_of_turn>model\n',
)

# ========== 학습 실행 ==========
print('⏱️ 학습 시작! (RTX 5060 Ti 기준 약 15~20분 소요)')
print('   학습이 진행되는 동안 logging_steps=10마다 Loss가 출력된다.')
print('   Loss가 점차 감소하면 학습이 잘 되고 있는 것이다.\n')

if torch.cuda.is_available():
    start_mem = torch.cuda.max_memory_reserved() / 1024**3
    print(f'학습 전 메모리: {start_mem:.2f} GB')

trainer_stats = trainer.train()

if torch.cuda.is_available():
    end_mem = torch.cuda.max_memory_reserved() / 1024**3
    print(f'\n학습 후 메모리: {end_mem:.2f} GB')
    print(f'학습에 사용된 메모리: {end_mem - start_mem:.2f} GB')

# [결과 해석]
# Loss: 값이 낮을수록 모델이 정답에 가깝게 학습된 것이다.
#   체스는 같은 상황에서 여러 좋은 수가 있으므로 0.5~0.8이면 충분하다.
# 메모리: 16GB 중 ~12GB 사용은 정상 범위이다.
#   QLoRA 덕분에 16GB GPU에서 학습이 가능했다는 것이 핵심이다.
print(f'\n학습 완료! Loss: {trainer_stats.training_loss:.4f}')
print(f'→ Loss가 0.5~0.8 범위이면 정상적으로 학습된 것이다.')
print(f'  (체스는 최선의 수가 여러 개이므로 Loss가 0에 수렴하지 않는다)')


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: Fine-tuning 전후 비교 — Before vs After
#
# [이 셀에서 하는 것]
# 챕터 1에서 기록한 학습 전 응답(response_before)과
# 학습 후 응답(response_after)을 나란히 비교한다.
#
# [확인 포인트]
# - 학습 전: 체스와 무관한 범용 답변 또는 엉뚱한 수 제시
# - 학습 후: "a6" 또는 그에 준하는 정확한 체스 수 예측
# - 정답: a6 (Morphy Defense — Ruy Lopez에서 흑의 대표적 응수)
#
# [이 결과가 의미하는 것]
# 전체 파라미터의 ~1%만 학습했는데도 체스 전문 지식이 생겼다.
# 16GB GPU에서 수십 분 만에 도메인 특화 모델을 만든 것이다.
# Full Fine-tuning(20GB+ 필요)으로는 아예 불가능했을 작업이
# QLoRA 덕분에 가능해진 것이 핵심이다.
# ═══════════════════════════════════════════════════════════

# 학습 후 추론 (같은 입력으로)
inputs = tokenizer(text, return_tensors='pt').to('cuda')
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)

response_after = tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
)

print('Fine-tuning 전후 비교')
print('=' * 60)
print(f'입력 기보: {test_input}')
print(f'(Ruy Lopez Opening — 흑의 최선 수는 a6)')
print(f'\n[학습 전] {response_before[:200]}')
print(f'\n[학습 후] {response_after[:200]}')
print(f'\n[정답] a6 (Morphy Defense)')

# 학습 효율 요약
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'\n── QLoRA 학습 효율 요약 ──')
print(f'  학습 파라미터: {trainable:,}개 (전체의 {trainable/total*100:.2f}%)')
print(f'  사용 GPU: 16GB VRAM (Full FT라면 20GB+ 필요)')
print(f'  핵심: 1%의 파라미터만 학습했지만, 체스 도메인 전문 지식 획득!')


---

## 6. 정리

### 오늘 배운 전체 흐름

```
① Fine-tuning이 필요한 이유 — 사전학습 모델은 특정 도메인에 약하다 (챕터 1)
② Full FT의 문제 — 모든 파라미터 학습 → 메모리 폭발 (챕터 2)
③ LoRA — 큰 행렬 대신 작은 행렬 두 개로 변화량 표현, 1%만 학습 (챕터 3)
④ QLoRA — 5-2의 양자화 + LoRA 결합 → 16GB GPU에서도 학습 가능 (챕터 4)
⑤ 실전 학습 — 데이터 변환 → Trainer → 학습 → 추론 비교 (챕터 5)
```

### 핵심 개념 요약

| 개념 | 한 줄 정리 |
|------|----------|
| **Fine-tuning** | 사전학습 모델을 특정 작업에 맞게 추가 학습 |
| **Full FT** | 모든 파라미터 학습. 성능 좋지만 메모리 폭발 |
| **PEFT** | 극소수 파라미터만 학습. 효율적 |
| **LoRA** | 저차원 행렬(B×A)로 변화량 표현. 전체의 ~1%만 학습 |
| **QLoRA** | 양자화(INT4) + LoRA. 16GB GPU에서 대형 모델 학습 가능 |
| **Unsloth** | 최적화된 QLoRA 프레임워크. 속도 30배↑, 메모리 30%↓ |

### 5-2(Quantization)와의 관계 정리

| 구분 | 5-2 (지난 시간) | 5-1 (오늘) |
|------|:---:|:---:|
| 목적 | **추론** 최적화 | **학습** 최적화 |
| 양자화 시점 | 추론 시 (PTQ) | 학습 시 (QLoRA의 Q) |
| LoRA | 사용 안 함 | **사용** (QLoRA의 LoRA) |
| 결과 | 가벼운 추론 모델 | **도메인 특화 모델** |
| 공통점 | 둘 다 **NF4 양자화** 사용 |

### 자기주도 실습 안내

이제 `실습_5-1_PEFT_파라미터_효율적_튜닝.ipynb`를 열고,
오늘 배운 개념을 TODO 코드로 직접 구현해 보자.


---

## AI 과정 정리 — 한 달간의 여정 돌아보기

### 우리가 걸어온 길

```
Chapter 2-1  토큰화와 임베딩        "컴퓨터는 텍스트를 어떻게 이해하는가?"
    ↓
Chapter 2-2  합성 데이터 생성        "학습 데이터가 부족하면 어떻게 하는가?"
    ↓
Chapter 3-1  CNN과 전이학습          "이미지를 인식하고, 학습된 지식을 재활용하는 법"
    ↓
Chapter 3-2  파운데이션 모델          "GPT, BERT는 어떻게 만들어졌는가?"
    ↓
Chapter 4-1  RAG                    "LLM에게 외부 지식을 연결하는 법"
    ↓
Chapter 4-2  AI Agent               "LLM이 스스로 판단하고 행동하게 만드는 법"
    ↓
Chapter 5-2  Quantization           "거대한 모델을 작은 GPU에서 돌리는 법"
    ↓
Chapter 5-1  PEFT (LoRA/QLoRA)      "적은 자원으로 모델을 내 도메인에 맞게 학습시키는 법"
```

이 여정은 단순히 개별 기술을 배운 것이 아니다.
`"AI 시스템을 설계하고 구축하는 전체 흐름"`을 경험한 것이다.

### 이 기술들로 만들 수 있는 것

배운 기술들을 조합하면 다음과 같은 시스템을 직접 구축할 수 있다.

| 프로젝트 예시 | 사용 기술 |
|-------------|----------|
| **사내 문서 Q&A 챗봇** | 임베딩(2-1) + RAG(4-1) + Agent(4-2) + Quantization(5-2) |
| **고객 서비스 자동화** | Agent(4-2) + Tool-use + 가드레일 + QLoRA(5-1)로 도메인 특화 |
| **전문 분야 AI 어시스턴트** | QLoRA(5-1)로 전문 데이터 학습 + TTP(5-2)로 품질 안정화 |
| **소규모 팀의 AI 서비스** | Quantization(5-2)으로 서버 비용 절감 + LoRA로 빠른 커스터마이징 |

핵심은 `"거대 기업만 AI를 만들 수 있는 시대는 끝났다"`는 것이다.
오픈소스 모델 + 양자화 + LoRA를 조합하면, 16GB GPU 하나로도
실용적인 AI 시스템을 구축할 수 있다는 것을 직접 경험했다.

### 앞으로의 학습 방향

지금까지 배운 것의 **자연스러운 다음 단계**를 정리하면:

| 관심 분야 | 다음 단계 | 배운 것과의 연결 |
|---------|---------|------|
| **AI 서비스 개발** | LangGraph 심화, MCP, A2A 프로토콜 | 4-1 RAG + 4-2 Agent 확장 |
| **모델 경량화/배포** | GGUF 변환, vLLM, TensorRT | 5-2 Quantization 심화 |
| **도메인 특화 모델** | DPO/RLHF, 데이터 큐레이션 | 5-1 QLoRA 심화 |
| **멀티모달 AI** | Vision-Language 모델, 이미지 생성 | 3-1 CNN + 3-2 파운데이션 확장 |

### 마무리

AI 기술은 빠르게 변하지만, 이 과정에서 배운 **기본 원리**는 쉽게 변하지 않는다.
토큰화, 임베딩, Attention, RAG, Agent, 양자화, LoRA —
이 개념들은 어떤 새로운 모델이 나와도 적용되는 **기초 체력**이다.


---

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성청년SW·AI아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.
